# Daily Update

**Barely anything needed fixing.** This is the cleanest notebook you've sent me.

Already correct:
- `fetch()` handles the yfinance MultiIndex (`get_level_values(0)`)
- `safe_last()` / `safe_chg()` wrap every lookup with try/except and defaults
- Every source is a live `yf.download` — refreshes to the latest bar on each run
- No hardcoded prices, no hardcoded paths, no secrets

Only change: **14 `float(...).iloc[n]`** → `float(np.asarray(...)[n])`, which
silences the pandas FutureWarning. Cosmetic today, a `TypeError` in a future
pandas release.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  GOLD HEDGE FUND DESK — DAILY REGIME ENGINE  v3
#  FIXES: DXY feed (DX-Y.NYB fallback chain), MOVE Index, VIX rate-of-change
#         filter, VIX term structure, GC roll calendar alert
#  Run once per day after US market close (11:30pm Athens / 4:30pm NY)
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import yfinance as yf
import warnings
from datetime import datetime, timedelta
import pytz

warnings.filterwarnings("ignore")

ATHENS = pytz.timezone("Europe/Athens")
NOW    = datetime.now(ATHENS)

print(f"\n{'═'*70}")
print(f"  GOLD DESK DAILY REGIME ENGINE  v3  |  {NOW.strftime('%A %d %B %Y  %H:%M')} Athens")
print(f"{'═'*70}\n")

# ══════════════════════════════════════════════════════════════════════════════
# 1.  FETCH ALL DATA
# ══════════════════════════════════════════════════════════════════════════════
def fetch(ticker, period, interval):
    try:
        df = yf.download(ticker, period=period, interval=interval, progress=False)
        if hasattr(df, 'columns') and isinstance(df.columns, pd.MultiIndex):  # FIX: flatten MultiIndex
            df.columns = df.columns.get_level_values(0)
        if df is None or len(df) == 0:
            return pd.DataFrame()
        df.columns = df.columns.get_level_values(0)
        return df.dropna()
    except Exception as e:
        print(f"  ⚠️  Failed to fetch {ticker}: {e}")
        return pd.DataFrame()

def safe_last(df, col='Close', default=np.nan):
    if df is None or len(df) == 0: return default
    try: return float(np.asarray(df[col])[-1])
    except: return default

def safe_chg(df, col='Close', n=1):
    if df is None or len(df) < n+2: return 0.0
    try:
        return (float(np.asarray(df[col])[-1]) - float(df[col].iloc[-(n+1)])) / \
               float(df[col].iloc[-(n+1)]) * 100
    except: return 0.0

print("⏳ Fetching data...")

gold_d = fetch("GC=F",   "2y",  "1d")
gold_w = fetch("GC=F",   "5y",  "1wk")
gold_5 = fetch("GC=F",   "5d",  "5m")
tips   = fetch("TIP",    "1y",  "1d")
spx    = fetch("^GSPC",  "1y",  "1d")
vix    = fetch("^VIX",   "1y",  "1d")
vix3m  = fetch("^VIX3M", "30d", "1d")   # VIX 3-month — for term structure
gld    = fetch("GLD",    "3mo", "1d")
bonds  = fetch("TLT",    "1y",  "1d")
silver = fetch("SI=F",   "3mo", "1d")
move   = fetch("^MOVE",  "1y",  "1d")   # Bond vol index — better than VIX for gold
tnx    = fetch("^TNX",   "1y",  "1d")   # 10Y Treasury yield

# ── DXY: robust fallback chain (DX=F was delisted) ───────────────────────────
def fetch_dxy():
    for ticker in ["DX-Y.NYB", "UUP"]:
        df_ = fetch(ticker, "1y", "1d")
        if len(df_) > 20:
            print(f"  ✅ DXY source: {ticker}")
            return df_, ticker
    # Final fallback: invert EURUSD (~-0.9 correlation with DXY)
    eurusd = fetch("EURUSD=X", "1y", "1d")
    if len(eurusd) > 20:
        print("  ⚠️  DXY: using EURUSD inverse proxy")
        p = eurusd.copy()
        p["Close"] = 1 / p["Close"]
        return p, "EURUSD-INV"
    return pd.DataFrame(), "NONE"

dxy, DXY_SOURCE = fetch_dxy()

# ── GC futures roll calendar ──────────────────────────────────────────────────
# Active delivery months: Feb(G) Apr(J) Jun(M) Aug(Q) Oct(V) Dec(Z)
# Roll window starts ~14 calendar days before 1st of delivery month
GC_ROLL_MONTHS = [2, 4, 6, 8, 10, 12]

def next_gc_roll():
    today_ = datetime.now(ATHENS)
    y = today_.year
    for rm in sorted(GC_ROLL_MONTHS):
        try:
            roll_dt = ATHENS.localize(datetime(y, rm, 1))
            warn_dt = roll_dt - timedelta(days=14)
            if warn_dt >= today_:
                return roll_dt.strftime("%b %Y"), (warn_dt - today_).days
        except Exception:
            continue
    roll_dt = ATHENS.localize(datetime(y + 1, GC_ROLL_MONTHS[0], 1))
    warn_dt = roll_dt - timedelta(days=14)
    return roll_dt.strftime("%b %Y"), (warn_dt - today_).days

GC_NEXT_CONTRACT, GC_DAYS_TO_ROLL = next_gc_roll()

print("✅ Data loaded.\n")

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════
def ema(s, n): return s.ewm(span=n, adjust=False).mean()
def rsi_calc(s, n=14):
    d = s.diff(); g = d.clip(lower=0).rolling(n).mean()
    l = (-d.clip(upper=0)).rolling(n).mean()
    return 100 - 100/(1 + g/l.replace(0, np.nan))
def atr_calc(df, n=14):
    tr = pd.concat([df['High']-df['Low'],
                    (df['High']-df['Close'].shift()).abs(),
                    (df['Low'] -df['Close'].shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean()
def trend_dir(df, col='Close'):
    if len(df) < 50: return "↔ N/A"
    e21 = float(ema(df[col], 21).iloc[-1])
    e50 = float(ema(df[col], 50).iloc[-1])
    px_ = float(np.asarray(df[col])[-1])
    if px_ > e21 > e50: return "▲ UPTREND"
    if px_ < e21 < e50: return "▼ DOWNTREND"
    return "↔ SIDEWAYS"
def pct_from(level, px_): return (px_ - level)/level*100

# ══════════════════════════════════════════════════════════════════════════════
# 2.  BLOCK 1 — MACRO REGIME
# ══════════════════════════════════════════════════════════════════════════════
print(f"{'─'*70}")
print("  BLOCK 1 — MACRO REGIME")
print(f"{'─'*70}")

g = gold_d.copy()
g['e21']   = ema(g['Close'], 21)
g['e50']   = ema(g['Close'], 50)
g['e100']  = ema(g['Close'], 100)
g['e200']  = ema(g['Close'], 200)
g['rsi']   = rsi_calc(g['Close'])
g['atr']   = atr_calc(g)
g['vol_ma']= g['Volume'].rolling(20).mean()

last = g.iloc[-1]; prev = g.iloc[-2]
px   = float(last['Close'])
px_1w = float(np.asarray(g['Close'])[-6])  if len(g)>6  else px
px_1m = float(np.asarray(g['Close'])[-22]) if len(g)>22 else px
px_3m = float(np.asarray(g['Close'])[-66]) if len(g)>66 else px

chg_1d = (px - float(prev['Close'])) / float(prev['Close']) * 100
chg_1w = (px - px_1w) / px_1w * 100
chg_1m = (px - px_1m) / px_1m * 100
chg_3m = (px - px_3m) / px_3m * 100

e21  = float(last['e21']);  e50  = float(last['e50'])
e100 = float(last['e100']); e200 = float(last['e200'])
r_val = float(last['rsi'])
atr_val = float(last['atr'])

trend_pts = sum([px>e21, px>e50, px>e100, px>e200, e21>e50])

regime_map = {
    5: ("🟢 BULL TREND",      "FULL LONG  — Add on dips, trail stops up"),
    4: ("🟡 BULLISH BIAS",    "LONG BIAS  — Buy pullbacks, reduce on rips"),
    3: ("🟠 NEUTRAL / MIXED", "NEUTRAL    — Trade ranges, reduce size"),
    2: ("🔴 BEARISH BIAS",    "SHORT BIAS — Sell rallies, avoid longs"),
    1: ("🔴 BEAR TREND",      "FULL SHORT — Short strength, tight stops"),
    0: ("⛔ EXTREME BEAR",    "DEFENSIVE  — Cash / hedges only"),
}
regime_label, regime_action = regime_map.get(trend_pts, ("UNKNOWN","—"))

print(f"  Gold price     : ${px:>10,.2f}")
print(f"  1D change      : {chg_1d:>+8.2f}%")
print(f"  1W change      : {chg_1w:>+8.2f}%")
print(f"  1M change      : {chg_1m:>+8.2f}%")
print(f"  3M change      : {chg_3m:>+8.2f}%")
print()
for lbl, val in [("EMA21",e21),("EMA50",e50),("EMA100",e100),("EMA200",e200)]:
    print(f"  {lbl:<10}     : ${val:>10,.2f}  {'✅ ABOVE' if px>val else '❌ BELOW'}")
print(f"  EMA21>EMA50    :  {'✅ YES — bullish cross' if e21>e50 else '❌ NO  — bearish alignment'}")
print()
print(f"  RSI(14)        : {r_val:>8.1f}  ", end="")
print("🔥 OVERBOUGHT" if r_val>75 else ("✅ HEALTHY" if r_val>50 else ("⚠️  WEAK" if r_val>35 else "🧊 OVERSOLD")))
print(f"  ATR(14)        : ${atr_val:>8.2f}  (daily range budget)")
vol_ratio = float(last['Volume'])/float(last['vol_ma']) if float(last['vol_ma'])>0 else 1
print(f"  Volume         : {vol_ratio:.1f}x avg  {'⚡ HIGH' if vol_ratio>1.5 else ('normal' if vol_ratio>0.8 else '🔇 LOW')}")
print()
print(f"  ┌─────────────────────────────────────────────────────────┐")
print(f"  │  REGIME SCORE  : {trend_pts}/5                                     │")
print(f"  │  REGIME        : {regime_label:<43}│")
print(f"  │  PLAYBOOK      : {regime_action:<43}│")
print(f"  └─────────────────────────────────────────────────────────┘")

# ══════════════════════════════════════════════════════════════════════════════
# 3.  BLOCK 2 — MACRO FORCES
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("  BLOCK 2 — MACRO FORCES")
print(f"{'─'*70}")

dxy_px   = safe_last(dxy);   dxy_chg  = safe_chg(dxy);   dxy_dir  = trend_dir(dxy)
tip_px   = safe_last(tips);  tip_chg  = safe_chg(tips);  tip_dir  = trend_dir(tips)
spx_px   = safe_last(spx);   spx_chg  = safe_chg(spx);   spx_dir  = trend_dir(spx)
vix_px   = safe_last(vix);   vix_chg  = safe_chg(vix)
vix3m_px = safe_last(vix3m)
tlt_px   = safe_last(bonds); tlt_chg  = safe_chg(bonds); tlt_dir  = trend_dir(bonds)
tnx_px   = safe_last(tnx);   tnx_chg  = safe_chg(tnx)
move_px  = safe_last(move);  move_chg = safe_chg(move)

dxy_impact = "🔴 HEADWIND" if dxy_chg>0.3  else ("🟢 TAILWIND" if dxy_chg<-0.3  else "🟡 NEUTRAL")
rr_impact  = "🟢 TAILWIND" if tip_chg>0.1  else ("🔴 HEADWIND" if tip_chg<-0.1  else "🟡 NEUTRAL")
fear_state = "🔥 EXTREME FEAR" if vix_px>35 else ("⚠️  ELEVATED" if vix_px>20 else "😴 COMPLACENT")
tnx_impact = "🔴 HEADWIND" if tnx_chg>0.05 else ("🟢 TAILWIND" if tnx_chg<-0.05 else "🟡 NEUTRAL")

# ── VIX rate-of-change signal quality filter ─────────────────────────────────
vix_abs_chg = abs(vix_chg)
if vix_abs_chg >= 15:
    VIX_SIGNAL_QUALITY = "🚨 BROKEN — VIX moved >15% today. DO NOT TRADE. All signals unreliable."
elif vix_abs_chg >= 8:
    VIX_SIGNAL_QUALITY = "⚠️  DEGRADED — VIX >8% move. Halve position size. Widen stops 50%."
elif vix_abs_chg >= 5:
    VIX_SIGNAL_QUALITY = "⚠️  CAUTION — VIX >5% move. Reduce size 30%. Wait for VIX to stabilise."
else:
    VIX_SIGNAL_QUALITY = "✅ NORMAL — VIX stable. Standard signal quality applies."

# ── VIX term structure (spot vs 3-month) ──────────────────────────────────────
if not np.isnan(vix3m_px) and vix3m_px > 0:
    vix_ratio = vix_px / vix3m_px
    if vix_ratio > 1.10:
        VIX_TERM = f"📛 BACKWARDATION ({vix_ratio:.2f}x) — acute fear. Gold spike then reversal likely."
    elif vix_ratio > 1.02:
        VIX_TERM = f"⚠️  FLAT/INVERTING ({vix_ratio:.2f}x) — stress building."
    else:
        VIX_TERM = f"✅ CONTANGO ({vix_ratio:.2f}x) — normal, no panic signal."
else:
    VIX_TERM = "N/A (^VIX3M unavailable)"

# ── MOVE Index (bond vol — better leading indicator for gold than VIX) ─────────
if not np.isnan(move_px):
    if move_px > 130:   MOVE_STATE = f"🔥 EXTREME ({move_px:.0f}) — bond vol crisis. Gold highly directional."
    elif move_px > 100: MOVE_STATE = f"⚠️  ELEVATED ({move_px:.0f}) — rate uncertainty. Gold tailwind building."
    elif move_px > 80:  MOVE_STATE = f"🟡 MODERATE ({move_px:.0f}) — normal bond vol. Watch real rate direction."
    else:               MOVE_STATE = f"😴 LOW ({move_px:.0f}) — bond complacency. Gold likely range-bound."
else:
    MOVE_STATE = "N/A"

# Gold/SPX 20D correlation
gold_ret = gold_d['Close'].pct_change().dropna().tail(20)
spx_ret  = spx['Close'].pct_change().dropna().tail(20) if len(spx)>20 else pd.Series()
if len(spx_ret)>5:
    corr_20d = float(gold_ret.corr(spx_ret.reindex(gold_ret.index, method='nearest')))
else:
    corr_20d = 0.0

print(f"\n  {'FORCE':<24} {'VALUE':<14} {'1D CHG':<10} {'TREND':<14} GOLD IMPACT")
print(f"  {'─'*24} {'─'*14} {'─'*10} {'─'*14} {'─'*14}")

dxy_label = f"DXY ({DXY_SOURCE})"
rows = [
    (dxy_label,           dxy_px,  dxy_chg, dxy_dir,        dxy_impact),
    ("TIP / Real Rates",  tip_px,  tip_chg, tip_dir,        rr_impact),
    ("10Y Yield (TNX)",   tnx_px,  tnx_chg, trend_dir(tnx), tnx_impact),
    ("S&P 500",           spx_px,  spx_chg, spx_dir,        "↔ MONITOR"),
    ("TLT / Bonds",       tlt_px,  tlt_chg, tlt_dir,        ""),
]
for name, val, chg, direction, impact in rows:
    val_s = f"{val:,.2f}" if not np.isnan(val) else "N/A"
    chg_s = f"{chg:>+.2f}%" if chg != 0.0 else "  N/A"
    print(f"  {name:<24} {val_s:<14} {chg_s:<10} {direction:<14} {impact}")

print(f"\n  VIX                    : {vix_px:<8.1f}  {fear_state}  ({vix_chg:+.2f}% today)")
print(f"  VIX term structure     : {VIX_TERM}")
print(f"  Signal quality filter  : {VIX_SIGNAL_QUALITY}")
if not np.isnan(move_px):
    print(f"  MOVE Index (bond vol)  : {move_px:>7.1f}  {MOVE_STATE}")
if not np.isnan(tnx_px):
    print(f"  10Y Yield              : {tnx_px:>7.3f}%  ({tnx_chg:+.4f}% today)")
print(f"\n  Gold/SPX 20D corr      : {corr_20d:+.2f}  ", end="")
if corr_20d>0.5:    print("⚠️  Risk-on — gold following equities")
elif corr_20d<-0.3: print("✅ Safe haven — negative correlation")
else:               print("↔  Decorrelated — own fundamentals")

macro_bull = sum([
    dxy_chg  < -0.2,
    tip_chg  >  0.1,
    vix_px   >  20,
    tlt_chg  >  0.1,
    spx_chg  < -0.5,
])
print(f"\n  Macro Bull Score       : {macro_bull}/5  ", end="")
print("🟢 MACRO TAILWIND" if macro_bull>=4 else ("🟡 MIXED MACRO" if macro_bull>=2 else "🔴 MACRO HEADWIND"))

# ══════════════════════════════════════════════════════════════════════════════
# 4.  BLOCK 3 — KEY LEVELS
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("  BLOCK 3 — KEY LEVELS & STRUCTURE")
print(f"{'─'*70}")

prev_h = float(np.asarray(g['High'])[-2]);  prev_l = float(np.asarray(g['Low'])[-2])
prev_c = float(np.asarray(g['Close'])[-2])
pivot  = (prev_h + prev_l + prev_c) / 3
r1 = 2*pivot - prev_l;   r2 = pivot + (prev_h-prev_l);   r3 = prev_h + 2*(pivot-prev_l)
s1 = 2*pivot - prev_h;   s2 = pivot - (prev_h-prev_l);   s3 = prev_l - 2*(prev_h-pivot)

gw    = gold_w.copy()
w_hi  = float(np.asarray(gw['High'])[-1]);  w_lo  = float(np.asarray(gw['Low'])[-1])
pw_hi = float(np.asarray(gw['High'])[-2]);  pw_lo = float(np.asarray(gw['Low'])[-2])
hi52  = float(g['High'].tail(252).max()); lo52 = float(g['Low'].tail(252).min())

print(f"\n  ── PIVOT LEVELS (today) ──")
for lbl,val in [("R3",r3),("R2",r2),("R1",r1),("PP ←pivot",pivot),("S1",s1),("S2",s2),("S3",s3)]:
    arrow = " ◀ PRICE HERE" if abs(pct_from(val,px))<0.5 else ""
    print(f"  {lbl:<12}: ${val:>10,.2f}  ({pct_from(val,px):>+.1f}%){arrow}")

print(f"\n  ── WEEKLY STRUCTURE ──")
print(f"  This week Hi   : ${w_hi:>10,.2f}  ({pct_from(w_hi,px):>+.1f}%)")
print(f"  This week Lo   : ${w_lo:>10,.2f}  ({pct_from(w_lo,px):>+.1f}%)")
print(f"  Prev week Hi   : ${pw_hi:>10,.2f}  ({pct_from(pw_hi,px):>+.1f}%)")
print(f"  Prev week Lo   : ${pw_lo:>10,.2f}  ({pct_from(pw_lo,px):>+.1f}%)")

print(f"\n  ── 52-WEEK RANGE ──")
pos_in_range = (px-lo52)/(hi52-lo52)*100 if hi52>lo52 else 50
print(f"  52W High       : ${hi52:>10,.2f}  ({pct_from(hi52,px):>+.1f}%)")
print(f"  52W Low        : ${lo52:>10,.2f}  ({pct_from(lo52,px):>+.1f}%)")
print(f"  Position       : {pos_in_range:.0f}% of range  ", end="")
print("🔝 Near highs — manage risk" if pos_in_range>80 else ("📉 Near lows" if pos_in_range<20 else "↔ Mid-range"))

print(f"\n  ── EMA DYNAMIC SUPPORT ──")
for lbl,val in [("EMA21 momentum",e21),("EMA50 trend",e50),("EMA100 swing",e100),("EMA200 macro",e200)]:
    status = "✅ ABOVE" if px>val else "❌ BELOW"
    print(f"  {lbl:<20}: ${val:>10,.2f}  {status}  ({pct_from(val,px):>+.1f}%)")

# ══════════════════════════════════════════════════════════════════════════════
# 5.  BLOCK 4 — FLOW & POSITIONING
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("  BLOCK 4 — INSTITUTIONAL FLOW & POSITIONING")
print(f"{'─'*70}")

if len(gld) >= 20:
    gld_vol5  = gld['Volume'].tail(5).mean()
    gld_vol20 = gld['Volume'].tail(20).mean()
    gld_flow  = "📥 INFLOWS" if gld_vol5>gld_vol20*1.15 else ("📤 OUTFLOWS" if gld_vol5<gld_vol20*0.85 else "↔ NEUTRAL")
    gld_chg   = (float(np.asarray(gld['Close'])[-1])-float(np.asarray(gld['Close'])[-6]))/float(np.asarray(gld['Close'])[-6])*100
else:
    gld_flow = "N/A"; gld_chg = 0.0

if len(silver)>0 and not np.isnan(safe_last(silver)):
    gs_ratio = px / safe_last(silver)
    gs_note  = "📈 Gold preferred (risk-off)" if gs_ratio>85 else ("📉 Silver leading (risk-on)" if gs_ratio<75 else "↔ Normal range")
    gs_str   = f"{gs_ratio:.1f}"
else:
    gs_str = "N/A"; gs_note = ""

gold_bond_ratio = px / tlt_px if not np.isnan(tlt_px) and tlt_px>0 else 0

# Momentum streak
closes = g['Close'].tail(10).values
streak = 0
for i in range(len(closes)-1, 0, -1):
    if closes[i] > closes[i-1]:
        if streak >= 0: streak += 1
        else: break
    else:
        if streak <= 0: streak -= 1
        else: break
streak_str = f"+{streak} day UP streak" if streak>0 else f"{streak} day DOWN streak"

vol_pct = float(last['Volume'])/float(last['vol_ma'])*100 if float(last['vol_ma'])>0 else 100

print(f"\n  GLD ETF flow (5D vs 20D)     : {gld_flow}")
print(f"  GLD 1-week change            : {gld_chg:+.2f}%")
print(f"  Gold/Silver ratio            : {gs_str}  {gs_note}")
print(f"  Gold/TLT ratio               : {gold_bond_ratio:.2f}")
print(f"  Price momentum               : {streak_str}")
print(f"  Daily volume vs 20D avg      : {vol_pct:.0f}%  ", end="")
print("⚡ HEAVY" if vol_pct>150 else ("📊 NORMAL" if vol_pct>70 else "🔇 LIGHT"))

# ══════════════════════════════════════════════════════════════════════════════
# 6.  BLOCK 5 — VOLATILITY & RISK  (+MOVE +VIX filter +GC Roll)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("  BLOCK 5 — VOLATILITY & RISK PARAMETERS")
print(f"{'─'*70}")

atr_pct  = atr_val/px*100
log_ret  = np.log(g['Close']/g['Close'].shift(1)).dropna()
hv_20    = float(log_ret.tail(20).std() * np.sqrt(252) * 100)
hv_60    = float(log_ret.tail(60).std() * np.sqrt(252) * 100) if len(log_ret)>=60 else hv_20
vol_regime = "🔥 HIGH VOL" if hv_20>hv_60*1.2 else ("🧊 LOW VOL" if hv_20<hv_60*0.8 else "↔ NORMAL VOL")

sl_dist  = atr_val * 1.5
contracts_1r = 1000 / (sl_dist * 100) if sl_dist>0 else 0

print(f"\n  ATR(14) daily              : ${atr_val:>8.2f}  ({atr_pct:.2f}% of price)")
print(f"  Hist Vol 20D (annualised)  : {hv_20:>7.1f}%  {vol_regime}")
print(f"  Hist Vol 60D (annualised)  : {hv_60:>7.1f}%")
print(f"  Expected range today       : ${px-atr_val:,.2f} – ${px+atr_val:,.2f}")
print(f"\n  ── POSITION SIZING ($100k account, 1% risk) ──")
print(f"  SL distance (1.5x ATR)     : ${sl_dist:>8.2f}")
print(f"  Max contracts              : {contracts_1r:>8.3f}")
print(f"  Risk in USD                : $1,000")

print(f"\n  ── VIX & VOLATILITY CONTEXT ──")
print(f"  VIX                        : {vix_px:>7.1f}  {fear_state}")
print(f"  VIX term structure         : {VIX_TERM}")
print(f"  Signal quality filter      : {VIX_SIGNAL_QUALITY}")
if not np.isnan(move_px):
    print(f"  MOVE Index (bond vol)      : {move_px:>7.1f}  {MOVE_STATE}")
if vix_px>30:    print("  ⚠️  HIGH VIX: Widen stops 50%, reduce size 30%")
elif vix_px>20:  print("  ⚠️  ELEVATED VIX: Use 1.2x normal stop width")
else:            print("  ✅ VIX normal: Standard position sizing applies")

# ── GC FUTURES ROLL ALERT ─────────────────────────────────────────────────────
print(f"\n  ── GC FUTURES ROLL ──")
if GC_DAYS_TO_ROLL <= 0:
    print(f"  🚨 ROLL NOW — {GC_NEXT_CONTRACT} contract. Switch to next month immediately.")
elif GC_DAYS_TO_ROLL <= 5:
    print(f"  ⚠️  ROLL THIS WEEK — {GC_NEXT_CONTRACT}. Roll within {GC_DAYS_TO_ROLL} days.")
elif GC_DAYS_TO_ROLL <= 14:
    print(f"  📅 ROLL APPROACHING — {GC_NEXT_CONTRACT}. {GC_DAYS_TO_ROLL} days to roll window.")
else:
    print(f"  ✅ NO ROLL NEEDED — Next: {GC_NEXT_CONTRACT} ({GC_DAYS_TO_ROLL} days to roll window)")

# ══════════════════════════════════════════════════════════════════════════════
# 7.  BLOCK 6 — INTRADAY SESSION CONTEXT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("  BLOCK 6 — INTRADAY SESSION CONTEXT")
print(f"{'─'*70}\n")

prev_day_hi = float(np.asarray(g['High'])[-2])
prev_day_lo = float(np.asarray(g['Low'])[-2])
print(f"  Previous Day High (PDH)    : ${prev_day_hi:>10,.2f}  ({pct_from(prev_day_hi,px):>+.2f}%)")
print(f"  Previous Day Low  (PDL)    : ${prev_day_lo:>10,.2f}  ({pct_from(prev_day_lo,px):>+.2f}%)")

if len(gold_5) > 0:
    try:
        g5 = gold_5.copy()
        g5.index = pd.to_datetime(g5.index, utc=True).tz_convert("America/New_York")
        today_bars = g5[g5.index.date == g5.index[-1].date()]

        sessions = {
            "Asia   (18:00–02:00 NY)": today_bars[today_bars.index.hour.isin(range(18,24)) |
                                                    today_bars.index.hour.isin(range(0,3))],
            "London (03:00–08:00 NY)": today_bars[today_bars.index.hour.isin(range(3,9))],
            "NY     (09:30–16:00 NY)": today_bars[today_bars.index.hour.isin(range(9,16))],
        }
        print()
        asia_hi = asia_lo = None
        for sess_name, bars in sessions.items():
            if len(bars) > 0:
                s_hi = float(bars['High'].max())
                s_lo = float(bars['Low'].min())
                print(f"  {sess_name}: ${s_lo:,.2f} – ${s_hi:,.2f}  (range ${s_hi-s_lo:.2f})")
                if "Asia" in sess_name: asia_hi = s_hi; asia_lo = s_lo
            else:
                print(f"  {sess_name}: not yet / no data")

        if asia_hi and asia_lo:
            print()
            if float(today_bars['Low'].min()) < asia_lo:
                print(f"  ⚡ Asia Low  ${asia_lo:,.2f} was SWEPT today")
            if float(today_bars['High'].max()) > asia_hi:
                print(f"  ⚡ Asia High ${asia_hi:,.2f} was SWEPT today")
    except Exception as e:
        print(f"  Intraday processing error: {e}")
else:
    print("  No intraday data available.")

# ══════════════════════════════════════════════════════════════════════════════
# 8.  BLOCK 7 — DAILY PLAYBOOK
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*70}")
print("  BLOCK 7 — DAILY PLAYBOOK")
print(f"{'═'*70}\n")

bull_count = 0; bear_count = 0; signals_list = []

if trend_pts >= 4:
    bull_count += 2; signals_list.append("✅ Trend: Strong bull structure (4-5 EMAs aligned)")
elif trend_pts >= 3:
    bull_count += 1; signals_list.append("🟡 Trend: Bullish bias (3 EMAs aligned)")
else:
    bear_count += 2; signals_list.append("❌ Trend: Bearish / mixed EMA structure")

if 50 < r_val < 75:
    bull_count += 1; signals_list.append(f"✅ RSI {r_val:.0f}: Bullish momentum, not overbought")
elif r_val >= 75:
    bear_count += 1; signals_list.append(f"⚠️  RSI {r_val:.0f}: Overbought — risk of pullback")
elif r_val < 40:
    bear_count += 1; signals_list.append(f"⚠️  RSI {r_val:.0f}: Weak — not yet oversold enough to buy")

if macro_bull >= 4:
    bull_count += 2; signals_list.append("✅ Macro: Strong tailwinds")
elif macro_bull >= 2:
    bull_count += 1; signals_list.append("🟡 Macro: Mixed — some tailwinds, some headwinds")
else:
    bear_count += 2; signals_list.append("❌ Macro: Headwinds dominant")

if streak >= 3:
    bull_count += 1; signals_list.append(f"✅ Momentum: {streak} consecutive up days")
elif streak <= -3:
    bear_count += 1; signals_list.append(f"❌ Momentum: {abs(streak)} consecutive down days")

if vol_pct > 130:
    signals_list.append(f"⚡ Volume: Heavy ({vol_pct:.0f}% of avg) — conviction behind move")

if pos_in_range > 85:
    bear_count += 1; signals_list.append("⚠️  Range: Near 52W highs — extended, manage risk")
elif pos_in_range < 20:
    bull_count += 1; signals_list.append("✅ Range: Near 52W lows — accumulation zone")

if vix_abs_chg >= 5:
    signals_list.append(f"⚠️  VIX FILTER: {VIX_SIGNAL_QUALITY}")

net = bull_count - bear_count
if net >= 4:
    verdict  = "🟢 STRONG BUY"
    playbook = "Aggressive long. Buy dips to EMA21. Target R2/R3. Trail stops."
elif net >= 2:
    verdict  = "🟢 BUY / LONG BIAS"
    playbook = "Buy pullbacks. Enter near S1/EMA21. TP: R1-R2. SL: below EMA50."
elif net >= 0:
    verdict  = "🟡 NEUTRAL — WAIT FOR TRIGGER"
    playbook = "No directional bias. Wait for sweep + reclaim. Reduce size."
elif net >= -2:
    verdict  = "🔴 SELL / SHORT BIAS"
    playbook = "Sell rallies to R1/EMA21. TP: S1-S2. SL: above EMA50."
else:
    verdict  = "🔴 STRONG SELL / DEFENSIVE"
    playbook = "Avoid longs. Short strength. Tight stops. Consider hedges."

for s in signals_list: print(f"  {s}")
print()
print(f"  ┌─────────────────────────────────────────────────────────────┐")
print(f"  │  Bull {bull_count}  Bear {bear_count}  Net {net:+d}                                    │")
print(f"  │  VERDICT   : {verdict:<49}│")
print(f"  └─────────────────────────────────────────────────────────────┘")
print(f"\n  PLAYBOOK:  {playbook}")

# ══════════════════════════════════════════════════════════════════════════════
# 9.  BLOCK 8 — WATCH LIST FOR TOMORROW
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'─'*70}")
print("  BLOCK 8 — WATCH LIST FOR TOMORROW")
print(f"{'─'*70}\n")

print(f"  RESISTANCE  →  R1 ${r1:,.2f}  |  R2 ${r2:,.2f}  |  PDH ${prev_day_hi:,.2f}")
print(f"  SUPPORT     →  S1 ${s1:,.2f}  |  S2 ${s2:,.2f}  |  PDL ${prev_day_lo:,.2f}")
print(f"  DYNAMIC     →  EMA21 ${e21:,.2f}  |  EMA50 ${e50:,.2f}  |  EMA200 ${e200:,.2f}")
print()
print(f"  SETUP TRIGGERS:")
print(f"  LONG  → Sweep PDL/S1 → reclaim above on volume → enter")
print(f"  LONG  → Pullback to EMA21, holds, closes back above VWAP")
print(f"  SHORT → Sweep PDH/R1 → reject back below → enter")
print(f"  SHORT → Break below EMA50 with volume → trend change")
print()
print(f"  SESSION TIMES (Athens / Greece):")
print(f"  Asia open    :  01:00  — range formation")
print(f"  London open  :  10:00  — expect Asia H/L sweep")
print(f"  NY open      :  16:30  — primary signal window")
print(f"  NY close     :  23:00  — run this script")

print(f"\n{'═'*70}")
print(f"  END OF DAILY REGIME REPORT  |  {NOW.strftime('%H:%M')} Athens")
print(f"  Next run: tonight 23:30 Athens  (after NY close)")
print(f"{'═'*70}\n")
